### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, the user is asking why parrots talk. Let me start by recalling what I know about parrots. Parrots are part of the Psittaciformes order, right? They’re known for their ability to mimic human speech, which is pretty unique among birds. But why exactly do they do that?\n\nFirst, I remember that parrots are highly social animals. They live in groups in the wild, so maybe talking is a way to communicate with their flock. But when they’re pets, they mimic humans instead. So, their social nature might be the root. They might talk to interact with their human companions, similar to how they interact with other parrots.\n\nAnother point is their intelligence. Parrots are among the most intelligent birds, with problem-solving skills and the ability to learn. Their mimicry could be a result of this intelligence, allowing them to adapt their communication methods. They might learn to talk by listening to their environment and repeating sounds they hear, like human

In [2]:
from langchain.tools import tool 

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': '6jbe4xk7z', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 154, 'total_tokens': 244, 'completion_time': 0.133614267, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time': 0.006813891, 'prompt_tokens_details': None, 'queue_time': 0.076179392, 'total_time': 0.140428158}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run-

### Tool Execution Loops


In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. A perfect day to enjoy outdoor activities! ☀️


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'ex4nqvwv6', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.142799629, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.007457467, 'prompt_tokens_details': None, 'queue_time': 0.072232362, 'total_time': 0.150257096}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'service_tier': 'on_deman